In [2]:
# --- FULL PIPELINE: SUNDARBANS FLOOD QUANTIFICATION & EXPORT ---

import ee
ee.Initialize()

print("1. Re-initializing variables and connecting to satellite...")
# 1. Background Variables (AOI, Dates, Data Fetching, and Masking)
sundarbans_aoi = ee.Geometry.Rectangle([88.0, 21.5, 90.5, 22.6])

# Fetch baseline and flood data
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(sundarbans_aoi) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .select('VH')

pre_flood_img = s1_collection.filterDate('2024-04-01', '2024-05-15').median().clip(sundarbans_aoi)
peak_flood_img = s1_collection.filterDate('2024-05-26', '2024-06-15').median().clip(sundarbans_aoi)

# Calculate the flood extent
pre_water = pre_flood_img.lt(-18)
peak_water = peak_flood_img.lt(-18)
flood_extent = peak_water.subtract(pre_water).gt(0)
flood_extent_masked = flood_extent.updateMask(flood_extent)

print("2. Variables locked. Calculating total flooded area...")
# 2. CALCULATE EXACT FLOODED AREA
flood_area_sq_m = flood_extent_masked.multiply(ee.Image.pixelArea())

flood_stats = flood_area_sq_m.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sundarbans_aoi,
    scale=10, 
    maxPixels=1e10
)

# Convert from square meters to square kilometers
flooded_sq_km = ee.Number(flood_stats.get('VH')).divide(1e6).round()
print(f"✅ SUCCESS! Total Flooded Area: {flooded_sq_km.getInfo()} Square Kilometers")

print("3. Triggering Cloud Export to Google Drive...")
# 3. EXPORT THE GEOTIFF TO GOOGLE DRIVE
export_task = ee.batch.Export.image.toDrive(
    image=flood_extent_masked,
    description='Sundarbans_CycloneRemal_Flood',
    folder='EarthEngine_Exports',
    fileNamePrefix='Sundarbans_Flood_Remal_2024',
    region=sundarbans_aoi,
    scale=10,
    maxPixels=1e10,
    crs='EPSG:4326' 
)

export_task.start()
print("✅ Export task successfully sent to Google's servers! Your GeoTIFF is building now.")

1. Re-initializing variables and connecting to satellite...
2. Variables locked. Calculating total flooded area...
✅ SUCCESS! Total Flooded Area: 1563 Square Kilometers
3. Triggering Cloud Export to Google Drive...
✅ Export task successfully sent to Google's servers! Your GeoTIFF is building now.


In [3]:
# Check what the Google servers are currently doing with your file
print(f"Current Status: {export_task.status()['state']}")

Current Status: RUNNING


In [4]:
# Check what the Google servers are currently doing with your file
print(f"Current Status: {export_task.status()['state']}")

Current Status: COMPLETED
